In [ ]:
import random
import numpy as np
import tqdm
import matplotlib.pyplot as plt
from mpl_toolkits import mplot3d
import os.path
import pandas as pd
from sklearn import preprocessing
import os

# User Inputs

In [ ]:
sample_size = 512 #enter the size to which events will be up/downsampled

# designed for data in the following format:
# x[0] ,y[1] ,z[2] ,time[3], Amplitude[4]
TRACK_CLASS = False
dimension = 4 # desired dimension of data to be input
ISOTOPE = 'O16'

In [ ]:
data = np.load('voxel_data/' + ISOTOPE + '_w_event_keys.npy')
event_lens = np.load('voxel_data/' + ISOTOPE + '_event_lens.npy')
data[56437,0,-1] = 56437
assert data.shape == (event_lens.size, np.max(event_lens), 6), 'Array has incorrect shape'
assert len(np.unique(data[:,:,5])) == event_lens.size, 'Array has incorrect Event_ids'

# Adding Labels
Before running code blocks below, put the `O16_labels.csv` file (get that file from Raghu or Michelle) into the `voxel_data` folder.

In [ ]:
df = pd.read_csv('voxel_data/O16_labels.csv')
labels = np.array(df['Number of tracks'])
indices = np.array(df['Event number'])
length = np.load('voxel_data/O16_event_lens.npy')

max_length = int(max(length))

types, distr = np.unique(labels, return_counts=True)
print('Total: ', types, distr)
size = np.sum(distr[0:-1])

dataset = np.zeros((len(indices), max_length, dimension + 3), float)
count = 0

for i in range(len(indices)):
    if not np.isnan(labels[i]):  # Ignore empty labels
        ev_num = int(indices[i])
        event_length = int(length[ev_num])  # Get the length of the current event

        # Copy data into the padded dataset
        dataset[count, :event_length, :3] = data[ev_num, :event_length, :3]  # xyz
        dataset[count, :event_length, 3] = data[ev_num, :event_length, 4]  # q

        dataset[count, 0, -3] = data[ev_num, 0, -1]  # Event index
        dataset[count, 0, -2] = labels[i]  # Number of tracks
        dataset[count, 0, -1] = length[ev_num]  # Length of event
        count += 1
print(dataset.shape)

# Filtering the Data

In [ ]:
def filter_point_clouds(data):
    
    # Extract only the x, y, z coordinates, and the charge (columns 0, 1, 2, 4)
    filtered_data = data[..., [0, 1, 2, 3]]
    return filtered_data

# Apply the filter function to the data
filtered_data = filter_point_clouds(dataset)

# Apply additional filtering based on the charge value
filtered_data[:,:,0] = np.where(filtered_data[:,:,3] < 80, 0, filtered_data[:,:,0])
filtered_data[:,:,1] = np.where(filtered_data[:,:,3] < 80, 0, filtered_data[:,:,1])
filtered_data[:,:,2] = np.where(filtered_data[:,:,3] < 80, 0, filtered_data[:,:,2])
filtered_data[:,:,3] = np.where(filtered_data[:,:,3] < 80, 0, filtered_data[:,:,3])

# Determine threshold for minimum number of non-zero points
min_points_threshold = 90

# Filter out events with too few non-zero points
filtered_events = []
valid_event_indices = []
for i in range(filtered_data.shape[0]):
    if np.count_nonzero(filtered_data[i, :, 0]) >= min_points_threshold:
        filtered_events.append(filtered_data[i])
        valid_event_indices.append(i)

filtered_data = np.array(filtered_events)

# Print the final filtered data for verification
print('Final Filtered Shape:', filtered_data.shape)
print(len(valid_event_indices))

# Random sample from NumPy array

In [ ]:
new_array_name = ISOTOPE + '_size' + str(sample_size) + '_sampled'
new_data = np.zeros((filtered_data.shape[0], sample_size, 7), float)
# Using tqdm with enumerate for progress indication
for idx, original_idx in tqdm.tqdm(enumerate(valid_event_indices), total=len(valid_event_indices)):
    # Filter out zero values using idx instead of original_idx
    non_zero_points = filtered_data[idx][filtered_data[idx, :, 0] != 0]
    non_zero_len = non_zero_points.shape[0]

    if original_idx == 56437:  # skipping the one empty event
        new_data[idx, 0, 5] = 56437
        continue

    if non_zero_len > sample_size:
        random_points = np.random.choice(non_zero_len, sample_size, replace=False)  # choosing the random instances to sample
        for count, r in enumerate(random_points):
            new_data[idx, count, :4] = non_zero_points[r, :4]  # Only use the filtered x, y, z
            #new_data[idx, count, 4:6] = dataset[original_idx, r, 5:]  # adding time from original data
            #new_data[idx, count, 4] = non_zero_points[r, 3]  # use amplitude (charge) from filtered data
            #new_data[idx, count, 5:] = dataset[original_idx, r, 5:]  # Add the remaining columns (event index) from the original data
    else:
        new_data[idx, :non_zero_len, :4] = non_zero_points[:, :4]  # Only use the filtered x, y, z
        #new_data[idx, :non_zero_len, 4:6] = dataset[original_idx, :non_zero_len, 5:]  # adding time from original data
        #new_data[idx, :non_zero_len, 4] = non_zero_points[:, 3]  # use amplitude (charge) from filtered data
        #new_data[idx, :non_zero_len, 5:] = dataset[original_idx, :non_zero_len, 5:]  # Add the remaining columns (event index) from the original data
        need = sample_size - non_zero_len
        random_points = np.random.choice(non_zero_len, need, replace=True if need > non_zero_len else False)
        for count, r in enumerate(random_points, start=non_zero_len):
            new_data[idx, count, :4] = non_zero_points[r, :4]  # Only use the filtered x, y, z
            #new_data[idx, count, 4:6] = dataset[original_idx, r, 5:]  # adding time from original data
            #new_data[idx, count, 4] = non_zero_points[r, 3]  # use amplitude (charge) from filtered data
            #new_data[idx, count, 5:] = dataset[original_idx, r, 5:]  # Add the remaining columns (event index) from the original data
    new_data[idx, 0, 4] = dataset[original_idx, 0, 4] # saving the event index
    new_data[idx, 0, 5] = dataset[original_idx, 0, 5]
    new_data[idx, 0, 6] = dataset[original_idx, 0, 6]

# Verify if there are still any zero values in new_data
print("Final check of new_data for zeros")
print("Number of zero values in x, y, z, charge columns:", np.count_nonzero(new_data[:, :, :4] == 0))
print("Indices of zero values:", np.where(new_data[:, :, :4] == 0))

# If there are still zeros, print a few samples where zeros are present
zero_indices = np.where(new_data[:, :, :4] == 0)
if len(zero_indices[0]) > 0:
    for i in range(min(5, len(zero_indices[0]))):
        print(f"Zero found at index {zero_indices[0][i]}, event {zero_indices[1][i]}, column {zero_indices[2][i]}")

assert np.all(new_data[:, :, :4] != 0), 'new_data contains zero values in the x, y, z, or charge columns'

np.save('voxel_data/' + new_array_name, new_data)

assert new_data.shape == (filtered_data.shape[0], sample_size, 7), 'Array has incorrect shape'
assert len(np.unique(new_data[:, :, 4])) == filtered_data.shape[0], 'Array has incorrect number of events'

In [ ]:
import numpy as np

def print_column_values(file_path):
    """Print values of the first column from the loaded data"""
    event_data = np.load(file_path)
    for n in range(10): #increase readability by just printing the first column of the first 10 events
    # for n in range(len(event_data)):
        column_values = event_data[n, 0, :]
        print(f'Values of first column:\n{column_values}')

print_column_values('voxel_data/O16_size512_sampled.npy')


# Classification

In [ ]:
def simplify_class(label):
    '''
    if int(label) == 0 or int(label) == 1 or int(label) == 2:
        return 0
    elif int(label) == 3:
        return 1
    else: # 4 or 5
        return 2
    '''
    if int(label) in [0, 1, 2, 3]:
        return int(label)
    else:
        return 4
    

new_data[:,0,-2] = list(map(simplify_class, new_data[:,0,-2]))
types, distr = np.unique(new_data[:,0,-2], return_counts=True)
print('After simplification: ', types, distr)
np.save('voxel_data/' + ISOTOPE + '_size' + str(sample_size), new_data)

In [ ]:
def print_column_values(file_path):
    """Print values of the first column from the loaded data"""
    event_data = np.load(file_path)
    for n in range(10):
    # for n in range(len(event_data)):
        column_values = event_data[n, 0, :]
        print(f'Values of first column:\n{column_values}')

print_column_values('voxel_data/O16_size512.npy')

# Scaling 

In [ ]:
# values correspond to the x,y,z,charge index
values = [0,1,2,3] 
means_and_stds = []
min_max_scaler = preprocessing.MinMaxScaler(feature_range=(-1, 1))
new_data[:,:,3] = np.log(new_data[:,:,3]) # log scale charge
# standard scaling 
for n in values:
    mean = np.mean(new_data[:,:,n])
    std = np.std(new_data[:,:,n])
    means_and_stds.append([mean,std])
    new_data[:,:,n] = (new_data[:,:,n] - mean) / std
new_data[:,0,-1] = np.log(new_data[:,0,-1])
new_data[:,0,-1] = min_max_scaler.fit_transform(new_data[:,0,-1].reshape(-1, 1)).reshape(1,-1)

assert np.sum(np.isnan(new_data)) == 0, 'NaNs in dataset'
assert np.sum(np.isinf(new_data)) == 0, 'Infinities in dataset'

# Split into training, testing, and validation sets
Performs a 60-20-20 split, including events at random.

In [ ]:
rand_shuffle = np.random.choice(len(new_data), len(new_data), replace = False)
name = ISOTOPE + '_size' + str(sample_size)

# 20-20 marking for test and validation
test_split = int(len(new_data) * .2)
val_split = int(len(new_data) * .4)

test = new_data[rand_shuffle[:test_split],:,:]
val = new_data[rand_shuffle[test_split:val_split],:,:]
train = new_data[rand_shuffle[val_split:],:,:]
print(len(new_data))
print(test.shape, val.shape, train.shape)

os.makedirs('data_splits/', exist_ok=True)
np.save('data_splits/' + ISOTOPE + '_size' + str(sample_size)+'_test', test)
np.save('data_splits/' + ISOTOPE + '_size' + str(sample_size)+'_val', val)
np.save('data_splits/' + ISOTOPE + '_size' + str(sample_size)+'_train', train)
assert len(np.unique(np.isnan(train[:,:,4]))) == 1, 'NaNs in dataset'
assert len(np.unique(np.isnan(val[:,:,4]))) == 1, 'NaNs in dataset'
assert len(np.unique(np.isnan(test[:,:,4]))) == 1, 'NaNs in dataset'

In [ ]:
train_rand_shuffle = np.random.choice(len(train), len(train), replace = False)
split= 0
train_data= train[train_rand_shuffle[split:],:,:]

val_rand_shuffle = np.random.choice(len(val), len(val), replace = False)
split= 0
val_data= val[val_rand_shuffle[split:],:,:]

test_rand_shuffle = np.random.choice(len(test), len(test), replace = False)
split= 0
test_data= test[test_rand_shuffle[split:],:,:]

train_f = train_data[:,:,:5]
train_l = train_data[:,0,5]

val_f = val_data[:,:,:5]
val_l = val_data[:,0,5]

test_f = test_data[:,:,:5]
test_l = test_data[:,0,5]

np.save("data_splits/O16_size512_train_features", train_f)
np.save("data_splits/O16_size512_train_labels", train_l)

np.save("data_splits/O16_size512_val_features", val_f)
np.save("data_splits/O16_size512_val_labels", val_l)

np.save("data_splits/O16_size512_test_features", test_f)
np.save("data_splits/O16_size512_test_labels", test_l)